# Home work no.5
>## professor : Dr. Vahid Khazaei nezhad
# Team Members :
- Saeed Tizkar
- Pooriya Amini
- Mohammad Elahi

## ???? ????? ?? online retail ???? ???? ?????? ???? ????? ??? ??? ????? ???? ? ????? ?????? ????

# Home work no.1
>## professor : Dr. Vahid Khazaei nezhad
# Team Members :
- Saeed Tizkar
- Pooriya Amini
- Mohammad Elahi# Importing Libraries

In [ ]:
import numpy as np
import pandas as pd

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

print('Libraries loaded.')

# Example 1: Simple use of algorithm

## Creating Dataset

In [ ]:
basket_lists = [['Milk', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
                ['Dill', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
                ['Milk', 'Apple', 'Kidney Beans', 'Eggs'],
                ['Milk', 'Unicorn', 'Corn', 'Kidney Beans', 'Yogurt'],
                ['Corn', 'Onion', 'Onion', 'Kidney Beans', 'Ice cream', 'Eggs']]

trans_encoder = TransactionEncoder()

encoded_baskets = trans_encoder.fit(basket_lists).transform(basket_lists)

dataset_df = pd.DataFrame(encoded_baskets, columns= trans_encoder.columns_)

dataset_df

## Computing support values

In [ ]:
apriori(dataset_df, min_support= 0.6, use_colnames= True)

# Example2: Association rule mining

## Loading dataset

In [ ]:
basket_df = pd.read_excel('./Online Retail.xlsx')

basket_df.head()

### Dataset description

## Some basic exploratory

In [ ]:
basket_df.shape

In [ ]:
basket_df.dtypes

In [ ]:
basket_df.isna().sum()

In [ ]:
canceled = basket_df.loc[basket_df['InvoiceNo'].str.contains('C') == True]

canceled.shape

## Cleansing dataset

In [ ]:
basket_df['Description'] = basket_df['Description'].str.strip()

#removing canceled records
basket_df.drop(canceled.index, inplace= True)

print(f'{canceled.shape[0]} has removed from dataset, new dataset size is {basket_df.shape[0]}')

In [ ]:
basket_df.loc[basket_df["Description"] == "x"]["Description"].empty

## Preparing dataset for analysing

### Selecting one region (Germany)

In [ ]:
france_basket = basket_df.loc[basket_df['Country'] == 'France']

france_basket.shape

### Creating transaction, items set

In [ ]:
def trans_basket_provider(in_basket):
    basket = []
    transaction_nom = []

    for transaction, items in in_basket.groupby('InvoiceNo').groups.items():

        products = []

        transaction_nom.append(transaction)
        for item in items:
            products.append(str(in_basket.loc[item]['StockCode']))        

        ###   
        basket.append(products)
    return (basket, transaction_nom)
fr_basket, fr_trans_nom = trans_basket_provider(france_basket)

### creating binary dataset

In [ ]:
def binary_df_provider(in_basket, trans_nom) :
    trans_encoder = TransactionEncoder()

    encoded_baskets = trans_encoder.fit(in_basket).transform(in_basket)
    binary_df = pd.DataFrame(encoded_baskets, columns= trans_encoder.columns_, index= trans_nom)

    return binary_df
fr_binary_df = binary_df_provider(fr_basket, fr_trans_nom)
fr_binary_df

### Association extraction(Recommendation based on antecedents)

In [ ]:
frequencies_items = apriori(fr_binary_df, min_support= 0.05, use_colnames= True)
frequencies_items
rules = association_rules(frequencies_items, metric= "lift", min_threshold= 1)

rules = rules.sort_values(['confidence', 'lift'], ascending= [False, False])

rules.reset_index(inplace= True, drop= True)

rules.head()

### Filtering results

In [ ]:
condition0 = rules.antecedents.apply(lambda x: False if 'POST' in x else True)

condition1 = rules.consequents.apply(lambda x: False if 'POST' in x else True)

condition2 = rules['antecedents'].apply(lambda x: len(x) >= 2)

rules.loc[condition0 & condition1 & condition2]

# Excersice1

In [ ]:
unique_stocks = basket_df.drop_duplicates(subset=["StockCode"])
def name_replacement (x) : 
    items = list(x)
    new_names = []
    for item in items:
        row = unique_stocks.loc[unique_stocks['StockCode'].apply(lambda y: str(y) == str(item))]
        if not row["Description"].empty:
            new_names.append(list(row["Description"])[0])
        else:
            new_names.append(item)
            
    return tuple(new_names)

In [ ]:
rules['antecedents'] = rules['antecedents'].apply(name_replacement)
print("antecedents column mapped")

In [ ]:
rules['consequents'] = rules['consequents'].apply(name_replacement)
print("consequents column mapped")

In [ ]:
rules.head()

In [ ]:
rules.loc[condition0 & condition1 & condition2]

# Excersize2

In [ ]:
swit_basket = basket_df.loc[basket_df['Country'] == 'Switzerland']
swit_basket

In [ ]:
sw_basket, sw_trans_nom = trans_basket_provider(swit_basket)

In [ ]:
sw_binary_df = binary_df_provider(sw_basket, sw_trans_nom)
sw_binary_df.head()

In [ ]:
sw_frequencies_items = apriori(sw_binary_df, min_support= 0.07, use_colnames= True)

sw_rules = association_rules(sw_frequencies_items, metric= "lift", min_threshold= 1)

sw_rules = sw_rules.sort_values(['confidence', 'lift'], ascending= [False, False])

sw_rules.reset_index(inplace= True, drop= True)

sw_rules.head()

In [ ]:
cond0 = sw_rules.antecedents.apply(lambda x: False if 'POST' in x else True)

cond1 = sw_rules.consequents.apply(lambda x: False if 'POST' in x else True)

cond2 = sw_rules['antecedents'].apply(lambda x: len(x) >= 2)

sw_rules.loc[cond0 & cond1 & cond2]

In [ ]:
sw_rules.corrwith(rules, numeric_only=True)